# Final RQ1 evaluation: does adding meal photos improve glucose forecasting? (held-out test, once)


In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [6]:
import json

import numpy as np
import pandas as pd
import torch

from src import dataloader as data
from src import metrics
from src import train
from src.models.classical_baseline import HORIZONS
from src.models.multimodal_residual import load_frozen_baseline, FrozenImageOnlyResidualPredictor

MODEL_NAME = 'CGM+Time+DINOv2 (frozen baseline + tuned image-only residual)'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]

ENCODER_NAME = 'dinov2'
IMG_FEAT_DIM = 768

# Locked hyperparameters -- selected entirely on validation data in
# tune_dinov2/tune_dinov2_residual.ipynb. Not re-tuned here.
IMG_PROJ_DIM = 8
IMG_DROPOUT = 0.5
LR = 3e-4
WEIGHT_DECAY = 0.0

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '6_best_image_encoder'
EXP2_CHECKPOINT_DIR = PROJECT_ROOT / 'experiments' / '3_multimodal' / 'multimodal' / 'checkpoints' / 'gated_cgm_time'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'final_rq1'
CHECKPOINT_DIR = EXPERIMENT_DIR / 'checkpoints' / 'final_rq1'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('Locked configuration:')
print(f'  encoder      = {ENCODER_NAME}')
print(f'  img_proj_dim = {IMG_PROJ_DIM}')
print(f'  img_dropout  = {IMG_DROPOUT}')
print(f'  lr           = {LR:.0e}')
print(f'  weight_decay = {WEIGHT_DECAY:.0e}')


Locked configuration:
  encoder      = dinov2
  img_proj_dim = 8
  img_dropout  = 0.5
  lr           = 3e-04
  weight_decay = 0e+00


In [4]:
stats = data.load_cgm_stats()

train_loader = data.load_multimodal_tensors('train', encoder_name=ENCODER_NAME)
val_loader = data.load_multimodal_tensors('val', encoder_name=ENCODER_NAME, shuffle=False)
test_loader = data.load_multimodal_tensors('test', encoder_name=ENCODER_NAME, shuffle=False)
test_participant_ids = data.load_multimodal_participant_ids('test')


print('n_train_batches:', len(train_loader))
print('n_val_batches:', len(val_loader))
print('n_test_batches:', len(test_loader))

sample_batch = next(iter(train_loader))
print('batch shapes:', [tuple(x.shape) for x in sample_batch])


def forward_fn(model, batch, device):
    cgm, tsm, has_meal, img_feat, target = batch
    pred = model(cgm.to(device), tsm.to(device), img_feat.to(device), has_meal.to(device))
    return pred, target.to(device)

n_train_batches: 323
n_val_batches: 64
n_test_batches: 66
batch shapes: [(256, 24), (256,), (256,), (256, 768), (256, 5)]


In [7]:
all_results = []

for seed in SEEDS:
    print(f"\n{'-' * 18} seed={seed} {'-' * 18}")

    base_model = load_frozen_baseline(seed, EXP2_CHECKPOINT_DIR)
    train.set_seed(seed)

    model = FrozenImageOnlyResidualPredictor(
        base_model=base_model, img_feat_dim=IMG_FEAT_DIM, img_proj_dim=IMG_PROJ_DIM,
        img_dropout=IMG_DROPOUT,
    )

    check_batch = next(iter(val_loader))
    cgm, tsm, has_meal, img_feat, _ = check_batch
    model.eval()
    base_model.eval()
    with torch.no_grad():
        base_pred = base_model(cgm, tsm, has_meal)
        residual_pred = model(cgm, tsm, img_feat, has_meal)
    init_diff = (base_pred - residual_pred).abs().max().item()
    print(f'initial max |residual - baseline| = {init_diff:.8f}')
    if init_diff > 1e-5:
        raise RuntimeError(
            f'seed={seed}: residual model does not match the frozen baseline at '
            f'initialisation (max diff={init_diff}).'
        )

    model, history = train.train_model(
        model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
        lr=LR, weight_decay=WEIGHT_DECAY, forward_fn=forward_fn,
        checkpoint_dir=CHECKPOINT_DIR / f'seed{seed}',
        verbose=False, progress_bar=True, progress_desc=f'seed={seed}',
    )

    y_pred_val, y_true_val = train.predict(model, val_loader, forward_fn=forward_fn)
    val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])

    # The one and only test evaluation for this configuration.
    y_pred_test, y_true_test = train.predict(model, test_loader, forward_fn=forward_fn)
    test_results = metrics.evaluate(y_pred_test, y_true_test, stats['cgm_mean'], stats['cgm_std'])
    test_results_per_participant = metrics.evaluate_per_participant(
        y_pred_test, y_true_test, test_participant_ids, stats['cgm_mean'], stats['cgm_std'],
    )

    val_results = {str(h): v for h, v in val_results.items()}
    test_results = {str(h): v for h, v in test_results.items()}
    test_results_per_participant = {
        str(pid): {str(h): v for h, v in per_h.items()}
        for pid, per_h in test_results_per_participant.items()
    }

    seed_result = {
        'model': MODEL_NAME, 'seed': seed, 'encoder': ENCODER_NAME,
        'img_proj_dim': IMG_PROJ_DIM, 'img_dropout': IMG_DROPOUT,
        'lr': LR, 'weight_decay': WEIGHT_DECAY,
        'best_epoch': len(history['train_loss']),
        'initial_max_abs_diff_vs_baseline': init_diff,
        'val_results': val_results,
        'test_results': test_results,
        'test_results_per_participant': test_results_per_participant,
    }
    all_results.append(seed_result)

    with open(RESULTS_DIR / f'final_rq1_dinov2_seed{seed}_results.json', 'w') as f:
        json.dump(seed_result, f, indent=2)

    test_rmse = [round(test_results[str(h)]['rmse'], 2) for h in HORIZONS]
    print(f'best epoch: {seed_result["best_epoch"]}  '
          f'test RMSE @ {HORIZONS} min = {test_rmse}')

print(f'\nCompleted final RQ1 evaluation for {len(all_results)} seeds.')


------------------ seed=7 ------------------
initial max |residual - baseline| = 0.00000000
device: cuda (NVIDIA A100-SXM4-40GB)  model: FrozenImageOnlyResidualPredictor  params: 1,032,642


seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

best epoch: 15  test RMSE @ [15, 30, 60, 90, 120] min = [10.76, 17.18, 25.86, 31.84, 35.38]

------------------ seed=14 ------------------
initial max |residual - baseline| = 0.00000000
device: cuda (NVIDIA A100-SXM4-40GB)  model: FrozenImageOnlyResidualPredictor  params: 1,032,642


seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

best epoch: 17  test RMSE @ [15, 30, 60, 90, 120] min = [10.68, 17.11, 26.02, 32.17, 35.75]

------------------ seed=21 ------------------
initial max |residual - baseline| = 0.00000000
device: cuda (NVIDIA A100-SXM4-40GB)  model: FrozenImageOnlyResidualPredictor  params: 1,032,642


...

best epoch: 14  test RMSE @ [15, 30, 60, 90, 120] min = [10.74, 17.34, 26.34, 32.55, 36.26]

------------------ seed=63 ------------------
initial max |residual - baseline| = 0.00000000
device: cuda (NVIDIA A100-SXM4-40GB)  model: FrozenImageOnlyResidualPredictor  params: 1,032,642


seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

best epoch: 14  test RMSE @ [15, 30, 60, 90, 120] min = [10.66, 17.13, 25.81, 31.83, 35.42]

------------------ seed=70 ------------------
initial max |residual - baseline| = 0.00000000
device: cuda (NVIDIA A100-SXM4-40GB)  model: FrozenImageOnlyResidualPredictor  params: 1,032,642


seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

best epoch: 14  test RMSE @ [15, 30, 60, 90, 120] min = [10.94, 17.34, 26.1, 32.1, 35.58]

Completed final RQ1 evaluation for 10 seeds.


In [9]:
# Load the two already-completed baselines' test results -- same population, same protocol,
# no retraining needed.
def load_seed_results(result_dir, stub, seeds, result_key='test_results'):
    loaded = {}
    for seed in seeds:
        path = result_dir / f'{stub}_seed{seed}_results.json'
        if not path.exists():
            raise FileNotFoundError(f'Missing result file for seed {seed}: {path}')
        with open(path) as f:
            result = json.load(f)
        if result_key not in result:
            raise KeyError(f'{path.name} missing "{result_key}". Available: {list(result.keys())}')
        loaded[seed] = result
    return loaded


def rmse_by_seed(loaded_results, seeds, horizons, result_key='test_results'):
    return {
        h: np.asarray([float(loaded_results[seed][result_key][str(h)]['rmse']) for seed in seeds],
                       dtype=float)
        for h in horizons
    }


CGM_ONLY_DIR = PROJECT_ROOT / 'experiments' / '3_multimodal' / 'multimodal' / 'results' / 'cgm_only'
CGM_TIME_DIR = PROJECT_ROOT / 'experiments' / '3_multimodal' / 'multimodal' / 'results' / 'gated_cgm_time'

cgm_only_loaded = load_seed_results(CGM_ONLY_DIR, 'cgm_only', SEEDS)
cgm_time_loaded = load_seed_results(CGM_TIME_DIR, 'gated_cgm_time', SEEDS)

cgm_only_by_h = rmse_by_seed(cgm_only_loaded, SEEDS, HORIZONS)
cgm_time_by_h = rmse_by_seed(cgm_time_loaded, SEEDS, HORIZONS)
dinov2_by_h = {
    h: np.asarray([float(r['test_results'][str(h)]['rmse']) for r in all_results], dtype=float)
    for h in HORIZONS
}

print(f'Loaded {len(cgm_only_loaded)} CGM-only, {len(cgm_time_loaded)} CGM+Time test seeds '
      f'(reused, not retrained), plus {len(all_results)} CGM+Time+DINOv2 seeds (this run).')


Loaded 10 CGM-only, 10 CGM+Time test seeds (reused, not retrained), plus 10 CGM+Time+DINOv2 seeds (this run).


In [10]:
# === RQ1 main result table: test RMSE, mean +/- sd across 10 seeds ===
print('=== RQ1 main result table (held-out test, mean +/- sd across 10 seeds) ===')
print(f"{'horizon':>10}{'CGM-only':>16}{'CGM+Time':>16}{'CGM+Time+DINOv2':>20}")

rq1_table = {}
for h in HORIZONS:
    co, ct, dv = cgm_only_by_h[h], cgm_time_by_h[h], dinov2_by_h[h]
    rq1_table[h] = {
        'cgm_only': {'mean': float(np.mean(co)), 'sd': float(np.std(co, ddof=1))},
        'cgm_time': {'mean': float(np.mean(ct)), 'sd': float(np.std(ct, ddof=1))},
        'cgm_time_dinov2': {'mean': float(np.mean(dv)), 'sd': float(np.std(dv, ddof=1))},
    }
    print(f"{h:>8} min"
          f"{np.mean(co):>10.2f} +/- {np.std(co, ddof=1):<4.2f}"
          f"{np.mean(ct):>10.2f} +/- {np.std(ct, ddof=1):<4.2f}"
          f"{np.mean(dv):>12.2f} +/- {np.std(dv, ddof=1):<4.2f}")

print()
print('=== Paired effect: CGM+Time+DINOv2 vs. CGM+Time (does the image add value?) ===')
print(f"{'horizon':>10}{'delta RMSE':>18}{'improvement %':>18}{'wins/seeds':>14}")
for h in HORIZONS:
    ct, dv = cgm_time_by_h[h], dinov2_by_h[h]
    delta = dv - ct
    improvement_pct = (ct - dv) / ct * 100.0
    wins = int(np.sum(dv < ct))
    print(f"{h:>8} min"
          f"{np.mean(delta):>11.4f} +/- {np.std(delta, ddof=1):<6.4f}"
          f"{np.mean(improvement_pct):>11.3f} +/- {np.std(improvement_pct, ddof=1):<6.3f}"
          f"{wins:>8}/{len(SEEDS)}")

print()
print('=== Paired effect: CGM+Time vs. CGM-only (does time_since_meal add value?) ===')
print(f"{'horizon':>10}{'delta RMSE':>18}{'improvement %':>18}{'wins/seeds':>14}")
for h in HORIZONS:
    co, ct = cgm_only_by_h[h], cgm_time_by_h[h]
    delta = ct - co
    improvement_pct = (co - ct) / co * 100.0
    wins = int(np.sum(ct < co))
    print(f"{h:>8} min"
          f"{np.mean(delta):>11.4f} +/- {np.std(delta, ddof=1):<6.4f}"
          f"{np.mean(improvement_pct):>11.3f} +/- {np.std(improvement_pct, ddof=1):<6.3f}"
          f"{wins:>8}/{len(SEEDS)}")


=== RQ1 main result table (held-out test, mean +/- sd across 10 seeds) ===
   horizon        CGM-only        CGM+Time     CGM+Time+DINOv2
      15 min     11.21 +/- 0.09     10.76 +/- 0.10       10.74 +/- 0.10
      30 min     18.52 +/- 0.09     17.27 +/- 0.09       17.21 +/- 0.09
      60 min     27.54 +/- 0.19     26.17 +/- 0.17       26.06 +/- 0.17
      90 min     33.00 +/- 0.20     32.20 +/- 0.23       32.17 +/- 0.24
     120 min     36.05 +/- 0.22     35.69 +/- 0.28       35.73 +/- 0.28

=== Paired effect: CGM+Time+DINOv2 vs. CGM+Time (does the image add value?) ===
   horizon        delta RMSE     improvement %    wins/seeds
      15 min    -0.0179 +/- 0.0077      0.167 +/- 0.073       10/10
      30 min    -0.0668 +/- 0.0240      0.387 +/- 0.139       10/10
      60 min    -0.1082 +/- 0.0732      0.413 +/- 0.280        9/10
      90 min    -0.0358 +/- 0.1153      0.111 +/- 0.359        7/10
     120 min     0.0394 +/- 0.1488     -0.111 +/- 0.419        5/10

=== Paired effect: 

In [11]:
# === Per-participant robustness check (test set) ===

def per_participant_rmse_summary(loaded_results, seeds, horizons, result_key='test_results_per_participant'):
    rows = []
    for h in horizons:
        per_participant_means = []
        for seed in seeds:
            pp = loaded_results[seed][result_key]
            per_participant_means.extend(v[str(h)]['rmse'] for v in pp.values())
        rows.append({
            'horizon_min': h,
            'mean_across_participants_and_seeds': float(np.mean(per_participant_means)),
            'sd_across_participants_and_seeds': float(np.std(per_participant_means, ddof=1)),
            'n_participant_seed_pairs': len(per_participant_means),
        })
    return pd.DataFrame(rows)


dinov2_loaded_for_pp = {r['seed']: r for r in all_results}

print('=== Per-participant test RMSE (mean/sd pooled across participants x seeds) ===')
for label, loaded in [('CGM-only', cgm_only_loaded), ('CGM+Time', cgm_time_loaded),
                       ('CGM+Time+DINOv2', dinov2_loaded_for_pp)]:
    df = per_participant_rmse_summary(loaded, SEEDS, HORIZONS)
    print(f'\n{label}:')
    print(df.round(3).to_string(index=False))


=== Per-participant test RMSE (mean/sd pooled across participants x seeds) ===

CGM-only:
 horizon_min  mean_across_participants_and_seeds  sd_across_participants_and_seeds  n_participant_seed_pairs
          15                              11.071                             1.745                        60
          30                              18.178                             3.568                        60
          60                              26.205                             8.503                        60
          90                              30.718                            12.116                        60
         120                              33.075                            14.409                        60

CGM+Time:
 horizon_min  mean_across_participants_and_seeds  sd_across_participants_and_seeds  n_participant_seed_pairs
          15                              10.637                             1.631                        60
          30               

In [12]:
# Save the RQ1 main result table.
SUMMARY_PATH = RESULTS_DIR / 'rq1_main_result_table.json'
with open(SUMMARY_PATH, 'w') as f:
    json.dump({
        'seeds': SEEDS,
        'locked_config': {
            'encoder': ENCODER_NAME, 'img_proj_dim': IMG_PROJ_DIM, 'img_dropout': IMG_DROPOUT,
            'lr': LR, 'weight_decay': WEIGHT_DECAY,
        },
        'test_rmse_by_horizon': {str(h): rq1_table[h] for h in HORIZONS},
    }, f, indent=2)

print(f'Saved RQ1 main result table to: {SUMMARY_PATH}')
print('\nThis is the final answer to RQ1: whether adding meal photographs (via a frozen')
print('CGM+time_since_meal baseline plus a tuned DINOv2 image-only residual correction)')
print('improves blood glucose forecasting, evaluated once on the held-out test set.')


Saved RQ1 main result table to: /content/drive/MyDrive/dissertation/experiments/6_best_image_encoder/results/final_rq1/rq1_main_result_table.json

This is the final answer to RQ1: whether adding meal photographs (via a frozen
CGM+time_since_meal baseline plus a tuned DINOv2 image-only residual correction)
improves blood glucose forecasting, evaluated once on the held-out test set.
